In [ ]:
""" Converts multiline animation sprite to a single line srpite. Generates the animation sequence for the scene file. """
import numpy as np
from PIL import Image
from pathlib import Path
base_dir = Path("../Assets/World/Buildings/Agricultural/CattleRun/Sprites")
src_image = Image.open(base_dir / "CattleRun_work_45.png")
tile_size = np.array([192, 192])
cols_rows = np.array([15, 10])
width_height = (tile_size[0]*cols_rows[0]*cols_rows[1], tile_size[1]) # one wide image
dst_image = Image.new("RGBA", tuple(width_height))
for i in range(0, 10):
  crop = np.array([0,0,tile_size[0]*cols_rows[0],tile_size[1]])
  crop += np.array([0,tile_size[1]*i,0,tile_size[1]*i])
  # display(crop)
  dst_image.paste(src_image.crop(tuple(crop)),(i*tile_size[0]*cols_rows[0],0))
dst_image.save(base_dir / "CattleRun_work_flat_45.png")
display(dst_image)

for i in range(cols_rows[0]*cols_rows[1]):
  print(f"0:0/animation_frame_{i}/duration = 1.0")

In [ ]:
# [ext_resource type="Texture2D" uid="uid://bvbq4ud7b5cx6" path="res://Assets/World/Buildings/Agricultural/CattleRun/Sprites/CattleRun_work_45.png" id="1_nncli"]
# generate atlas textures:
for y in range(0, 15):
	for x in range(0, 20):
		src_resource_id = "1_nncli"
		x_start = x*192
		x_end = (x+1)*192
		y_start = y*192
		y_end = (y+1)*192
		print(
f"""
[sub_resource type="AtlasTexture" id="AtlasTexture_{x:02}{y:02}"]
atlas = ExtResource("1_nncli")
region = Rect2({x_start}, {y_start}, {x_end}, {y_end})""")

for y in range(0, 15):
	for x in range(0, 20):
		print(
f"""{{
"duration": 1.0,
"texture": SubResource("AtlasTexture_{x:02}{y:02}")
}},"""
		)


In [ ]:
import os
# godot_parser - godot 3 file format only? doesn't support godot 4 animations
from godot_parser import *
from pathlib import Path

# Create a new TSCN file
file_content = Path("../Assets/World/Buildings/Agricultural/CattleRun/CattleRun2D.tscn").read_text()
scene = GDScene.parse(file_content)
scene.write(Path("../Assets/World/Buildings/Agricultural/CattleRun/CattleRun2D.2.tscn"))

In [ ]:
from pathlib import Path
from jinja2 import Environment, FileSystemLoader
project_root: Path = Path("S:/src/Richard/Godot/unknown-horizon-godot")
images_path: Path = project_root / "Assets/World/Components/Collectors/BuildingCollector/Sprites"

Assets\World\Components\Collectors\BuildingCollector\Sprites


In [31]:
files: list = []

for state_folder in images_path.glob("*"): # go through the folders with the collector states
  if state_folder.is_dir(): # if a state folder
    state_list: list = [] # the list for the state
    for rotation_folder in state_folder.glob("*"): # go through the folders with the rotations
      if rotation_folder.is_dir() and rotation_folder.stem.isnumeric(): # if a rotation folder
        png_file_list = list(rotation_folder.glob("*.png")) # get the png files
        state_list.append(png_file_list) # add the list of pngs to the folder containing the rotations
    files.append(state_list) # add the list of rotations to the list of states

[[[WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/0/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/135/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/180/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/225/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/270/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/315/0000.png')], [WindowsPath('S:/src/Richard/Godot/unknown-horizon-godot/Assets/World/Components/Collectors/BuildingCollector/Sprites/idle/45/0000.png')], [WindowsPath('S:/src/

In [107]:
sprite_frames_tres_template = """
[gd_resource type="SpriteFrames" load_steps=1{#let the godot engine correct it#} format=3 uid = "uid://{{images_folder[0][0][0].parent.parent.parent.parent.stem}}"]
{% for state in images_folder -%}
  {%- for rotation in state%}
    {%- for image in rotation%}
[ext_resource type="Texture2D" uid="" path="res://{{image.relative_to("S:/src/Richard/Godot/unknown-horizon-godot").as_posix() }}" id="{{tier + "_" + image.parent.parent.stem + "_" + image.parent.stem + "_" + image.stem}}"]
    {%- endfor %}
  {%- endfor %}
{%- endfor %}

[resource]
animations = [{
"frames": [{
"duration": 1.0,
"texture": null
}],
"loop": true,
"name": &"Empty",
"speed": 5.0
},
{%- for state in images_folder %}
  {%- for rotation in state%}
{
"frames": [
    {%- for image in rotation%}
{
"duration": 1.0,
"texture": ExtResource("{{tier + "_" + image.parent.parent.stem + "_" + image.parent.stem + "_" + image.stem}}")
},
    {%- endfor %}
],
"loop": true,
"name": "{{tier + "_" + rotation[0].parent.parent.stem + "_" + rotation[0].parent.stem}}",
"speed": 5.0
},
  {%- endfor %}
{%- endfor %}
]
"""

In [50]:
def write_template(output_path: Path, template_str: str, args: dict):
  print(f"Writing template: {output_path}")
  env = Environment(loader=FileSystemLoader("."))
  template = env.from_string(template_str.strip())
  rendered_content = template.render(args)
  output_path.write_text(rendered_content)


In [108]:
write_template(project_root / "Assets/World/Components/Collectors/BuildingCollector/Sprites/BuildingCollectorFrames.tres", sprite_frames_tres_template, {"images_folder": files, "tier": "sailors"})

Writing template: S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites\BuildingCollectorFrames.tres
